In [ ]:
import os,sys,json,hashlib,zipfile,io,subprocess,shutil
from pathlib import Path
from datetime import datetime,timezone
RUN_DIR=Path('/kaggle/working/cure_cpu')/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True,exist_ok=False)
SOURCE=RUN_DIR/'source'
payload=__import__('base64').b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='9559b4b04f4b95f7f4e8da11a5fcea14ffa0a9248c6ee742c41febd706519abb'
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert z.testzip() is None
    assert all(not n.startswith('/') and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(SOURCE)
sys.path.insert(0,str(SOURCE))


In [ ]:
import glob
os.environ.update(TRANSFORMERS_NO_TF='1',TRANSFORMERS_NO_FLAX='1',TRANSFORMERS_OFFLINE='1',HF_HUB_OFFLINE='1',LITELLM_LOCAL_MODEL_COST_MAP='True',LITELLM_TELEMETRY='False',OTEL_SDK_DISABLED='true',PYTHONIOENCODING='utf-8')
for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth','/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for path in glob.glob(pattern): Path(path).unlink(missing_ok=True)
wheels=list(Path('/kaggle/input').rglob('swegemma-0.2.11-py3-none-any.whl'))
assert len(wheels)==1
wheelhouse=wheels[0].parent
links=RUN_DIR/'wheelhouse_links'; links.mkdir()
for p in wheelhouse.glob('*.whl'):
    if 'cutlass' in p.name.lower(): continue
    name=p.name.replace('cu128','+cu128') if 'cu128' in p.name and '+' not in p.name else p.name
    (links/name).symlink_to(p)
pins=['swegemma-0.2.11-py3-none-any.whl','adk_submission-0.2.13-py3-none-any.whl','adk_eval_core-0.1.0-py3-none-any.whl','google_adk-1.36.1-py3-none-any.whl','google_genai-2.11.0-py3-none-any.whl']
try:
    with (RUN_DIR/'installation.log').open('x') as log:
        # Match official Getting Started: curated wheels, no dependency resolver.
        subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--force-reinstall']+
            sorted(str(p) for p in links.glob('*.whl')),stdout=log,stderr=subprocess.STDOUT,check=True,timeout=1800)
finally:
    shutil.rmtree(links)


In [ ]:
try:
    p=subprocess.run([sys.executable,'-m','cure.cpu_validation','cpu-r03',str(RUN_DIR/'validation')],cwd=SOURCE,capture_output=True,text=True,timeout=1200)
    (RUN_DIR/'runner.stdout.log').write_text(p.stdout)
    (RUN_DIR/'runner.stderr.log').write_text(p.stderr)
    print(p.stdout,flush=True)
    assert p.returncode==0,'CPU validation failed; no GPU launch authorized'
finally:
    print('CURE_CPU_EVIDENCE',shutil.make_archive(str(RUN_DIR),'zip',root_dir=RUN_DIR),flush=True)
